In [9]:
import datetime
from collections import Counter
import numpy as np
import qnexus as qnx
from pytket import Circuit
from pytket.circuit.display import render_circuit_jupyter,get_circuit_renderer



def calculate_hamming_distance_pdf(data: dict) -> tuple[list, list]:
    """
    Calculates the hamming distance between number of bitstrings and
    ground state |00..0> .

    Parameters:
    data (dict):  Contains bitstring as key, count of bitstring as item

    Returns:
    tuple: The


    """
    distances = Counter()
    for key, item in data.items():
        dist = np.sum(key)
        distances.update({dist: item})

    xs = sorted(distances.keys())
    total = sum(distances.values())
    pdf = [distances[x] / total for x in xs]
    return xs, pdf


def validate_nexus_connection(nexus_hosted=True) -> bool:
    """
    Returns True if we can auth and reach Nexus (and therefore can submit jobs).

    Parameters:
    nexus_hosted (bool) : Checks that local machine connected to nexus hosted api

    Returns:
    bool: Success of connection (true).

    """
    try:
        # (https://docs.quantinuum.com/nexus/trainings/notebooks/basics/getting_started.html
        qnx.login_with_credentials()  # prompt-based

        # This will fail if tokens are invalid, network is blocked, etc.
        df = qnx.devices.get_all(nexus_hosted=nexus_hosted).df()
        return (df is not None) and (len(df) > 0)

    except ConnectionError as ce:
        raise ConnectionError("Failed to connect to quantinuum machine...") from ce


_PERT_UNITARY_OPS = {"X", "Y", "Z", "H", "S", "T"}
_INIT_PAULIS = ["X", "Y", "Z"]  # identity handled by skipping

_PERT_UNITARY_OPS = {"X", "Y", "Z", "H", "S", "T"}
_INIT_PAULIS = ["X", "Y", "Z"]  # identity handled by skipping


def generate_time_reversal_breaking_random_brick_wall(
    L: int,
    W: int,
    coherent: bool = False,
    pert_site: int | None = None,
    pert_op: str = "measure",
    probe_site: int | None = None,
    probe_angle: float = 0.5,
    add_barrier: bool = True,
    do_reset: bool = False,
    seed: int | None = None,
    p: float = 0.0,
) -> Circuit:
    """
    Docstring for generate_random_brick_wall_echo_puncture_after_U

    Parameters:
    L (int): The length of circuit or number of layers.
    W (int): The width of the circuit or number of qubits.
    coherent (bool): When True the mid-circuit perturbation is suppressed; stochastic
    measurements controlled by p are unaffected.
    pert_site (int): Qubit index where the mid-circuit perturbation is inserted.
    Defaults to W // 2.
    pert_op (str): Operation applied at pert_site when coherent=False.
    Use "measure" for a projective measurement or any single-qubit gate name
    supported by pytket Circuit (e.g. "X", "Y", "Z", "H", "S", "T").
    probe_site (int): Qubit prepared in an equal superposition before U and
    measured after U†. Must differ from pert_site. Defaults to 0.
    probe_angle (float): Ry rotation angle (in pytket half-turns, i.e. units of π)
    applied to probe_site during state preparation. 0.5 → Ry(π/2), -0.5 → Ry(-π/2).
    do_reset (bool): After a "measure" perturbation, reset the qubit to |0>.
    p (float): Probability in [0, 1] of inserting a projective measurement on
    each qubit after each unitary block. Applied symmetrically in both U and U†.

    Returns:
    Circuit: State-prep → U → Perturbation → U† → probe measurement circuit.

    """

    if pert_op != "measure" and pert_op not in _PERT_UNITARY_OPS:
        raise ValueError(
            f"Unsupported pert_op {pert_op!r}. Use 'measure' or one of {_PERT_UNITARY_OPS}."
        )

    rng = np.random.default_rng(seed=seed)

    if pert_site is None:
        pert_site = W // 2
    if probe_site is None:
        probe_site = 0
    if probe_site == pert_site:
        raise ValueError(
            f"probe_site and pert_site must differ (both are {probe_site})."
        )

    def layer_pairs(l):
        if l % 2 == 0:
            return [(2 * w, 2 * w + 1) for w in range(W // 2)]
        else:
            return [(2 * w + 1, 2 * w + 2) for w in range(W // 2 - 1)]

    # Pre-generate all gate parameters and stochastic measurement decisions so
    # the total classical bit count is known before circuit construction.
    forward = []
    for l in range(L):
        for i, j in layer_pairs(l):
            a, b, g = rng.normal(size=3)
            ai, bi, ci = rng.uniform(0, 2, size=3)  # TK1 angles for qubit i
            aj, bj, cj = rng.uniform(0, 2, size=3)  # TK1 angles for qubit j
            allow_meas = p > 0 and 1 <= l <= L - 2
            meas_i = bool(allow_meas and rng.random() < p)
            meas_j = bool(allow_meas and rng.random() < p)
            forward.append((a, b, g, i, j, ai, bi, ci, aj, bj, cj, meas_i, meas_j))

    # bit 0       : mid-circuit perturbation (if pert_op == "measure")
    # bits 1..N   : U stochastic measurements  (N = n_p_meas)
    # bits N+1..2N: U† stochastic measurements
    # bit 2N+1    : final probe measurement
    n_p_meas = sum(mi + mj for *_, mi, mj in forward)
    n_bits = 2 + 2 * n_p_meas
    qc = Circuit(W, n_bits)

    barrier_sequence = list(range(W))
    all_bits = list(range(n_bits))
    bit_idx_u = 1
    bit_idx_ud = 1 + n_p_meas
    probe_bit = 1 + 2 * n_p_meas

    # --- State preparation ---
    # probe_site → equal superposition via Ry; all other sites → random {I, X, Y, Z}.
    qc.Ry(probe_angle, probe_site)
    for q in range(W):
        if q == probe_site:
            continue
        op = rng.choice(["I"] + _INIT_PAULIS)
        if op != "I":
            getattr(qc, op)(q)

    # Build U: unitary block (TK2, TK1_i, TK1_j) then stochastic measurements.
    for a, b, g, i, j, ai, bi, ci, aj, bj, cj, meas_i, meas_j in forward:
        qc.TK2(a, b, g, i, j)
        qc.TK1(ai, bi, ci, i)
        qc.TK1(aj, bj, cj, j)
        if meas_i:
            qc.Measure(i, bit_idx_u)
            bit_idx_u += 1
        if meas_j:
            qc.Measure(j, bit_idx_u)
            bit_idx_u += 1

    if add_barrier:
        qc.add_barrier(qubits=barrier_sequence, bits=all_bits)

    # Mid-circuit perturbation (toggled off when coherent=True).
    if not coherent:
        if pert_op == "measure":
            qc.Measure(pert_site, 0)
            if do_reset:
                qc.Reset(pert_site)
        else:
            getattr(qc, pert_op)(pert_site)

    if add_barrier:
        qc.add_barrier(qubits=barrier_sequence, bits=all_bits)

    # Apply U†: inverse of the unitary block is TK1_j† → TK1_i† → TK2†, then measurements.
    for a, b, g, i, j, ai, bi, ci, aj, bj, cj, meas_i, meas_j in reversed(forward):
        qc.TK1(-cj, -bj, -aj, j)
        qc.TK1(-ci, -bi, -ai, i)
        qc.TK2(-a, -b, -g, i, j)
        if meas_j:
            qc.Measure(j, bit_idx_ud)
            bit_idx_ud += 1
        if meas_i:
            qc.Measure(i, bit_idx_ud)
            bit_idx_ud += 1

    # Final probe measurement.
    qc.Measure(probe_site, probe_bit)

    return qc


def debug_after_measurement(qc: Circuit, n: int = 20) -> None:
    """
    Auxiliary function which checks that measurement is at
    middle of the time process i.e. between U and U^dagger.

    qc (Circuit): Circuit to be investigated
    n (int): Number of layers in system
    """

    cmds = qc.get_commands()

    # find the (first) Measure command
    m_idx = None
    for idx, cmd in enumerate(cmds):
        if cmd.op.type.name == "Measure":
            m_idx = idx
            break
    if m_idx is None:
        raise RuntimeError("No Measure found in circuit")

    print("MEASURE at command index:", m_idx)
    print("Next commands:")
    for k in range(m_idx + 1, min(m_idx + 1 + n, len(cmds))):
        cmd = cmds[k]
        optype = cmd.op.type.name
        qbs = [q.index[0] for q in cmd.qubits]  # qubit indices
        print(f"{k:4d}  {optype:10s}  qubits={qbs}")


def count_consecutive_tk2_after_measure(qc):
    """
    Docstring for count_consecutive_tk2_after_measure

    :param qc: Description
    """

    cmds = qc.get_commands()
    m_idx = next(i for i, c in enumerate(cmds) if c.op.type.name == "Measure")
    cnt = 0
    pairs = []
    for cmd in cmds[m_idx + 1 :]:
        if cmd.op.type.name != "TK2":
            break
        cnt += 1
        qbs = [q.index[0] for q in cmd.qubits]
        pairs.append(tuple(qbs))
    return cnt, pairs


def main(
    L: int, W: int, n_shots: int = 100, device_name: str = "H2-1E"
) -> "ExecutionResults":
    """
    Main function which runs experiment of U Measurement U^dagger

    L (int): Length of the circuit U
    W (int): Number of qubits.
    n_shots (int) : Number of times run the circuit.
    device_name (str): Quantinuum device name.

    Returns:


    """

    ok = validate_nexus_connection(nexus_hosted=True)
    print("Nexus connection OK?", ok)

    # submit job part
    incoherent_qc = generate_time_reversal_breaking_random_brick_wall(
        L, W, coherent=False
    )

    incoherent_qc.measure_all()

    my_project_ref = qnx.projects.get_or_create(
        name="measurement-induced butterfly effect"
    )

    # upload circuit
    circuit_ref = qnx.circuits.upload(
        circuit=incoherent_qc,
        name=f"{datetime.datetime.now()}_butterfly_incoherent_circuit_ref",
        project=my_project_ref,
    )

    # compile circuit
    compile_job_ref = qnx.start_compile_job(
        programs=[circuit_ref],
        name=f"{datetime.datetime.now()}_butterfly_incoherent_compiled_circuits",
        optimisation_level=1,
        backend_config=qnx.QuantinuumConfig(device_name=device_name),
        project=my_project_ref,
    )

    qnx.jobs.wait_for(compile_job_ref)

    compiled_circuits = [
        item.get_output() for item in qnx.jobs.results(compile_job_ref)
    ]

    # execute circuit
    collapsed_circuit = qnx.execute(
        programs=compiled_circuits,
        name=f"{datetime.datetime.now()}_butterfly_incoherent_execution",
        n_shots=n_shots,
        backend_config=qnx.QuantinuumConfig(device_name=device_name),
        project=my_project_ref,
        timeout=10000,
    )

    return collapsed_circuit




In [11]:
qc= generate_time_reversal_breaking_random_brick_wall(2,4,pert_op="X" ,p=.5)
render_circuit_jupyter(qc)

In [17]:
W = 5

def layer_pairs(l):
    if l % 2 == 0:
        return [(2 * w, 2 * w + 1) for w in range(W // 2)]
    else:
        return [(2 * w + 1, 2 * w + 2) for w in range(W // 2 - 1)]
    

layer_pairs(10)

[(0, 1), (2, 3)]

In [ ]:
W = 4
L = 4 
rng = np.random.default_rng(seed=10)
meas_rng = np.random.default_rng(seed=11)
p = .5 

forward = []
for l in range(4):
    print(l)
    for i, j in layer_pairs(l):
        print(i,j)
        a, b, g = rng.normal(size=3)
        ai, bi, ci = rng.uniform(0, 2, size=3)  # TK1 angles for qubit i
        aj, bj, cj = rng.uniform(0, 2, size=3)  # TK1 angles for qubit j
        # u_i, u_j are always drawn so the RNG sequence (and gate angles)
        # are identical for every p; p is a pure threshold on fixed values.
        u_i, u_j = meas_rng.random(), meas_rng.random()
        print(u_i,u_j)
        allow_meas = 1 <= l <= L - 2
        meas_i = bool(allow_meas and u_i < p)
        print(meas_i)
        meas_j = bool(allow_meas and u_j < p)
        print(meas_j)
        forward.append((a, b, g, i, j, ai, bi, ci, aj, bj, cj, meas_i, meas_j))

0
0 1
0.12857020276919962 0.49927786244011496
False
False
2 3
0.6014983576233575 0.028689008371944547
False
False
1
1 2
0.14792608457745593 0.9282110229603695
True
False
2
0 1
0.07042057615419683 0.12977394939929798
True
True
2 3
0.9483284532917751 0.6218835927963828
False
False
3
1 2
0.368993123729791 0.5113900218032627
False
False


In [21]:
forward = []
for l in range(L):
        pairs_in_layer = layer_pairs(l)
        paired_qubits = {q for pair in pairs_in_layer for q in pair}
        unpaired_qubits = [q for q in range(W) if q not in paired_qubits]
        allow_meas = 1 <= l <= L - 2

        pairs_data = []
        for i, j in pairs_in_layer:
            a, b, g = rng.normal(size=3)
            ai, bi, ci = rng.uniform(0, 2, size=3)  # TK1 angles for qubit i
            aj, bj, cj = rng.uniform(0, 2, size=3)  # TK1 angles for qubit j
            # u_i, u_j are always drawn so the RNG sequence (and gate angles)
            # are identical for every p; p is a pure threshold on fixed values.
            u_i, u_j = meas_rng.random(), meas_rng.random()
            meas_i = bool(allow_meas and u_i < p)
            meas_j = bool(allow_meas and u_j < p)
            pairs_data.append((a, b, g, i, j, ai, bi, ci, aj, bj, cj, meas_i, meas_j))

        singletons_data = []
        for q in unpaired_qubits:
            u_q = meas_rng.random()
            meas_q = bool(allow_meas and u_q < p)
            singletons_data.append((q, meas_q))

        forward.append((pairs_data, singletons_data))

In [22]:
forward

[([(np.float64(-0.3448165176946628),
    np.float64(-0.3923426027377626),
    np.float64(0.5898959165637815),
    0,
    1,
    np.float64(1.4773413500720114),
    np.float64(1.156909985198042),
    np.float64(0.9014711335151284),
    np.float64(0.5420488352125077),
    np.float64(1.7292062932359624),
    np.float64(0.137311331837066),
    False,
    False),
   (np.float64(-1.05242308652736),
    np.float64(-0.11729315387593296),
    np.float64(0.6750199304550224),
    2,
    3,
    np.float64(1.6664586282537301),
    np.float64(0.6820334262009562),
    np.float64(1.0395830138905775),
    np.float64(1.098412898650966),
    np.float64(0.38575436526213736),
    np.float64(0.6664337727139638),
    False,
    False)],
  []),
 ([(np.float64(0.8046315867376995),
    np.float64(-0.5641675681698397),
    np.float64(-0.3444514077774843),
    1,
    2,
    np.float64(1.367662437810518),
    np.float64(1.0472262358875581),
    np.float64(0.9287451390228469),
    np.float64(1.017258720029617),
   

In [44]:
def generate_time_reversal_breaking_random_brick_wall(
    L: int,
    W: int,
    unperturbed: bool = False,
    pert_site: int | None = None,
    pert_op: str = "measure",
    probe_site: int | None = None,
    probe_angle: float = 0.5,
    add_barrier: bool = True,
    do_reset: bool = False,
    seed: int | None = None,
    init_seed: int | None = None,
    meas_seed: int | None = None,
    p: float = 0.0,
) -> Circuit:
    """
    Docstring for generate_random_brick_wall_echo_puncture_after_U

    Parameters:
    L (int): The length of circuit or number of layers.
    W (int): The width of the circuit or number of qubits.
    unperturbed (bool): When True the mid-circuit perturbation is suppressed; stochastic
    measurements controlled by p are unaffected.
    pert_site (int): Qubit index where the mid-circuit perturbation is inserted.
    Defaults to W // 2.
    pert_op (str): Operation applied at pert_site when unperturbed=False.
    Use "measure" for a projective measurement or any single-qubit gate name
    supported by pytket Circuit (e.g. "X", "Y", "Z", "H", "S", "T").
    probe_site (int): Qubit prepared in an equal superposition before U and
    measured after U†. Must differ from pert_site. Defaults to 0.
    probe_angle (float): Ry rotation angle (in pytket half-turns, i.e. units of π)
    applied to probe_site during state preparation. 0.5 → Ry(π/2), -0.5 → Ry(-π/2).
    do_reset (bool): After a "measure" perturbation, reset the qubit to |0>.
    p (float): Probability in [0, 1] of inserting a projective measurement on
    each qubit after each unitary block. Applied symmetrically in both U and U†.

    Returns:
    Circuit: State-prep → U → Perturbation → U† → probe measurement circuit.

    """

    if pert_op != "measure" and pert_op not in _PERT_UNITARY_OPS:
        raise ValueError(
            f"Unsupported pert_op {pert_op!r}. Use 'measure' or one of {_PERT_UNITARY_OPS}."
        )

    if W % 2 != 0:
        raise ValueError("Circuit only defined for even # of qubits.")

    rng = np.random.default_rng(seed=seed)
    init_rng = np.random.default_rng(seed=init_seed if init_seed is not None else seed)
    meas_rng = np.random.default_rng(seed=meas_seed if meas_seed is not None else seed)

    if pert_site is None:
        pert_site = W // 2
    if probe_site is None:
        probe_site = 0
    if probe_site == pert_site:
        raise ValueError(
            f"probe_site and pert_site must differ (both are {probe_site})."
        )

    def layer_pairs(l):
        if l % 2 == 0:
            return [(2 * w, 2 * w + 1) for w in range(W // 2)]
        else:
            return [(2 * w + 1, 2 * w + 2) for w in range(W // 2 - 1)]

    # Pre-generate all gate parameters and stochastic measurement decisions so
    # the total classical bit count is known before circuit construction.
    # forward[l] = (pairs_data, singletons_data) where singletons_data covers
    # qubits not in any pair that layer (boundary qubits 0, W-1 in odd layers),
    # ensuring every qubit gets an independent measurement draw each layer so
    # the empirical rate converges to p uniformly across all W sites.
    forward = []
    for l in range(L):
        pairs_in_layer = layer_pairs(l)
        paired_qubits = {q for pair in pairs_in_layer for q in pair}
        unpaired_qubits = [q for q in range(W) if q not in paired_qubits]
        allow_meas = l <= L - 1

        pairs_data = []
        for i, j in pairs_in_layer:
            a, b, g = rng.normal(size=3)
            ai, bi, ci = rng.uniform(0, 2, size=3)  # TK1 angles for qubit i
            aj, bj, cj = rng.uniform(0, 2, size=3)  # TK1 angles for qubit j
            # u_i, u_j are always drawn so the RNG sequence (and gate angles)
            # are identical for every p; p is a pure threshold on fixed values.
            u_i, u_j = meas_rng.random(), meas_rng.random()
            meas_i = bool(allow_meas and u_i < p)
            meas_j = bool(allow_meas and u_j < p)
            pairs_data.append((a, b, g, i, j, ai, bi, ci, aj, bj, cj, meas_i, meas_j))

        singletons_data = []
        for q in unpaired_qubits:
            u_q = meas_rng.random()
            meas_q = bool(allow_meas and u_q < p)
            singletons_data.append((q, meas_q))

        forward.append((pairs_data, singletons_data))

    # bit 0 : scratch — receives every mid-circuit measurement (outcomes discarded)
    # bit 1 : probe — final readout only
    n_bits = 2
    probe_bit = 1
    qc = Circuit(W, n_bits)

    barrier_sequence = list(range(W))
    all_bits = list(range(n_bits))

    # --- State preparation ---
    # probe_site → equal superposition via Ry; all other sites → random {I, X, Y, Z}.
    qc.Ry(probe_angle, probe_site)
    for q in range(W):
        if q == probe_site:
            continue
        op = init_rng.choice(["I"] + _INIT_PAULIS)
        if op != "I":
            getattr(qc, op)(q)

    # Build U: unitary block (TK2, TK1_i, TK1_j) then stochastic measurements.
    # All mid-circuit outcomes are written to the scratch bit (0); they are not
    # read back, so overwriting is safe and keeps the classical register minimal.
    for pairs_data, singletons_data in forward:
        for a, b, g, i, j, ai, bi, ci, aj, bj, cj, meas_i, meas_j in pairs_data:
            qc.TK2(a, b, g, i, j)
            qc.TK1(ai, bi, ci, i)
            qc.TK1(aj, bj, cj, j)
            if meas_i:
                qc.Measure(i, 0)
            if meas_j:
                qc.Measure(j, 0)
        for q, meas_q in singletons_data:
            if meas_q:
                qc.Measure(q, 0)

        if add_barrier:
            qc.add_barrier(qubits=barrier_sequence, bits=all_bits)

    # Mid-circuit perturbation (toggled off when unperturbed=True).
    if not unperturbed:
        if pert_op == "measure":
            qc.Measure(pert_site, 0)
            if do_reset:
                qc.Reset(pert_site)
        else:
            getattr(qc, pert_op)(pert_site)

        if add_barrier:
            qc.add_barrier(qubits=barrier_sequence, bits=all_bits)

    # Apply U†: inverse of the unitary block is TK1_j† → TK1_i† → TK2†, then measurements.
    for pairs_data, singletons_data in reversed(forward):
        for q, meas_q in reversed(singletons_data):
            if meas_q:
                qc.Measure(q, 0)
        for a, b, g, i, j, ai, bi, ci, aj, bj, cj, meas_i, meas_j in reversed(pairs_data):
            qc.TK1(-cj, -bj, -aj, j)
            qc.TK1(-ci, -bi, -ai, i)
            qc.TK2(-a, -b, -g, i, j)
            if meas_j:
                qc.Measure(j, 0)
            if meas_i:
                qc.Measure(i, 0)

    # Final probe measurement.
    qc.Measure(probe_site, probe_bit)

    return qc

In [47]:
qc = generate_time_reversal_breaking_random_brick_wall(5,4,pert_op="X",p=.5)

In [48]:
render_circuit_jupyter(qc)

In [49]:
def generate_time_reversal_breaking_random_brick_wall(
    L: int,
    W: int,
    unperturbed: bool = False,
    pert_site: int | None = None,
    pert_op: str = "measure",
    probe_site: int | None = None,
    probe_angle: float = 0.5,
    add_barrier: bool = True,
    do_reset: bool = False,
    seed: int | None = None,
    init_seed: int | None = None,
    meas_seed: int | None = None,
    n_meas_total: int = 0,
) -> Circuit:
    """
    Parameters:
    L (int): The length of circuit or number of layers.
    W (int): The width of the circuit or number of qubits.
    unperturbed (bool): When True the mid-circuit perturbation is suppressed;
    stochastic measurements are unaffected.
    pert_site (int): Qubit index where the mid-circuit perturbation is inserted.
    Defaults to W // 2.
    pert_op (str): Operation applied at pert_site when unperturbed=False.
    Use "measure" for a projective measurement or any single-qubit gate name
    supported by pytket Circuit (e.g. "X", "Y", "Z", "H", "S", "T").
    probe_site (int): Qubit prepared in an equal superposition before U and
    measured after U†. Must differ from pert_site. Defaults to 0.
    probe_angle (float): Ry rotation angle (in pytket half-turns, i.e. units of π)
    applied to probe_site during state preparation. 0.5 → Ry(π/2), -0.5 → Ry(-π/2).
    do_reset (bool): After a "measure" perturbation, reset the qubit to |0>.
    n_meas_total (int): Exact number of stochastic mid-circuit measurements to
    insert across the entire U (and mirrored in U†). They are spread as
    uniformly as possible across the L-1 eligible layers (first through
    second-to-last); within each layer the measured qubits are chosen
    uniformly at random without replacement via meas_rng.

    Returns:
    Circuit: State-prep → U → Perturbation → U† → probe measurement circuit.

    """

    if pert_op != "measure" and pert_op not in _PERT_UNITARY_OPS:
        raise ValueError(
            f"Unsupported pert_op {pert_op!r}. Use 'measure' or one of {_PERT_UNITARY_OPS}."
        )

    if W % 2 != 0:
        raise ValueError("Circuit only defined for even # of qubits.")

    rng = np.random.default_rng(seed=seed)
    init_rng = np.random.default_rng(seed=init_seed if init_seed is not None else seed)
    meas_rng = np.random.default_rng(seed=meas_seed if meas_seed is not None else seed)

    if pert_site is None:
        pert_site = W // 2
    if probe_site is None:
        probe_site = 0
    if probe_site == pert_site:
        raise ValueError(
            f"probe_site and pert_site must differ (both are {probe_site})."
        )

    def layer_pairs(l):
        if l % 2 == 0:
            return [(2 * w, 2 * w + 1) for w in range(W // 2)]
        else:
            return [(2 * w + 1, 2 * w + 2) for w in range(W // 2 - 1)]

    # Distribute n_meas_total measurements across the L-1 eligible layers
    # (layers 0..L-2; the last layer is excluded because it sits immediately
    # before the perturbation). Each eligible layer gets floor(M/(L-1)) or
    # ceil(M/(L-1)) measurements; the remainder is assigned to a random subset
    # of layers drawn from meas_rng. Within each layer the measured qubits are
    # chosen uniformly at random without replacement, also via meas_rng.
    num_eligible = max(L - 1, 0)
    layer_meas_count = [0] * L
    if num_eligible > 0 and n_meas_total > 0:
        clamped = min(n_meas_total, W * num_eligible)
        base, remainder = divmod(clamped, num_eligible)
        for rank, l in enumerate(meas_rng.permutation(num_eligible)):
            layer_meas_count[l] = base + (1 if rank < remainder else 0)

    # Pre-generate gate parameters and measurement site selections so the total
    # classical bit count is known before circuit construction begins.
    forward = []
    for l in range(L):
        pairs_in_layer = layer_pairs(l)
        paired_qubits = {q for pair in pairs_in_layer for q in pair}
        unpaired_qubits = [q for q in range(W) if q not in paired_qubits]

        n_l = min(layer_meas_count[l], W)  # can't measure more qubits than exist
        meas_qubits = set(meas_rng.choice(W, size=n_l, replace=False).tolist()) if n_l > 0 else set()

        pairs_data = []
        for i, j in pairs_in_layer:
            a, b, g = rng.normal(size=3)
            ai, bi, ci = rng.uniform(0, 2, size=3)  # TK1 angles for qubit i
            aj, bj, cj = rng.uniform(0, 2, size=3)  # TK1 angles for qubit j
            pairs_data.append((a, b, g, i, j, ai, bi, ci, aj, bj, cj, i in meas_qubits, j in meas_qubits))

        singletons_data = [(q, q in meas_qubits) for q in unpaired_qubits]

        forward.append((pairs_data, singletons_data))

    # bit 0 : scratch — receives every mid-circuit measurement (outcomes discarded)
    # bit 1 : probe — final readout only
    n_bits = 2
    probe_bit = 1
    qc = Circuit(W, n_bits)

    barrier_sequence = list(range(W))
    all_bits = list(range(n_bits))

    # --- State preparation ---
    # probe_site → equal superposition via Ry; all other sites → random {I, X, Y, Z}.
    qc.Ry(probe_angle, probe_site)
    for q in range(W):
        if q == probe_site:
            continue
        op = init_rng.choice(["I"] + _INIT_PAULIS)
        if op != "I":
            getattr(qc, op)(q)

    # Build U: unitary block (TK2, TK1_i, TK1_j) then stochastic measurements.
    # All mid-circuit outcomes are written to the scratch bit (0); they are not
    # read back, so overwriting is safe and keeps the classical register minimal.
    # A barrier after each layer keeps the circuit diagram readable; disable via
    # add_barrier=False for local simulators that reject barriers on classical bits.
    for pairs_data, singletons_data in forward:
        for a, b, g, i, j, ai, bi, ci, aj, bj, cj, meas_i, meas_j in pairs_data:
            qc.TK2(a, b, g, i, j)
            qc.TK1(ai, bi, ci, i)
            qc.TK1(aj, bj, cj, j)
            if meas_i:
                qc.Measure(i, 0)
            if meas_j:
                qc.Measure(j, 0)
        for q, meas_q in singletons_data:
            if meas_q:
                qc.Measure(q, 0)
        if add_barrier:
            qc.add_barrier(qubits=barrier_sequence, bits=all_bits)

    # Mid-circuit perturbation (toggled off when unperturbed=True).
    if not unperturbed:
        if pert_op == "measure":
            qc.Measure(pert_site, 0)
            if do_reset:
                qc.Reset(pert_site)
        else:
            getattr(qc, pert_op)(pert_site)

    if add_barrier:
        qc.add_barrier(qubits=barrier_sequence, bits=all_bits)

    # Apply U†: inverse of the unitary block is TK1_j† → TK1_i† → TK2†, then measurements.
    for pairs_data, singletons_data in reversed(forward):
        for q, meas_q in reversed(singletons_data):
            if meas_q:
                qc.Measure(q, 0)
        for a, b, g, i, j, ai, bi, ci, aj, bj, cj, meas_i, meas_j in reversed(pairs_data):
            qc.TK1(-cj, -bj, -aj, j)
            qc.TK1(-ci, -bi, -ai, i)
            qc.TK2(-a, -b, -g, i, j)
            if meas_j:
                qc.Measure(j, 0)
            if meas_i:
                qc.Measure(i, 0)
        if add_barrier:
            qc.add_barrier(qubits=barrier_sequence, bits=all_bits)

    # Final probe measurement.
    qc.Measure(probe_site, probe_bit)

    return qc

In [54]:
qc = generate_time_reversal_breaking_random_brick_wall(5,4,pert_op="X",n_meas_total=int(.5*5*4))

In [55]:
render_circuit_jupyter(qc)

In [57]:
from pytket.extensions.quantinuum import QuantinuumBackend

for d in QuantinuumBackend.available_devices():
    print(d.device_name)

H1-1
H1-1LE
H2-1
H2-1LE
H2-2
H2-2LE


In [59]:
qnx.login()

qnx.devices.get_all().df()


🌐 Browser log in initiated.


╭────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                        │
│         Confirm that the browser shows the following code and click 'allow device':    │
│                                                                                        │
│                                      j9vvL2                                            │
│                                                                                        │
╰────────────────────────────────────────────────────────────────────────────────────────╯

Browser didn't open automatically? Use this link: https://nexus.quantinuum.com/auth/device/browser?otp=j9vvL2gPu6YITQFJYmn427McK5IKUuAsZOfRGbcq6USxmkeCQxPTc9cWopm-EvNdyXIASjRDTOvT0Ija1Hkt0A
✅ Successfully logged in as dlakhdar@umd.edu using the browser.


,backend_name,device_name,nexus_hosted,backend_info
0,Aer,aer_simulator,True,"BackendInfo(name='AerBackend', device_name='ae..."
1,AerState,aer_simulator_statevector,True,"BackendInfo(name='AerStateBackend', device_nam..."
2,AerUnitary,aer_simulator_unitary,True,"BackendInfo(name='AerUnitaryBackend', device_n..."
3,Braket,sv1,True,"BackendInfo(name='BraketBackend', device_name=..."
4,Helios-1E-lite,None,True,"BackendInfo(name='Helios-1E-lite', device_name..."
5,Quantinuum,H2-2E,False,BackendInfo(name='EmulatorEnabledQuantinuumBac...
6,Quantinuum,H2-1E,False,BackendInfo(name='EmulatorEnabledQuantinuumBac...
7,Quantinuum,H2-1SC,False,BackendInfo(name='EmulatorEnabledQuantinuumBac...
8,Quantinuum,H2-1,False,BackendInfo(name='EmulatorEnabledQuantinuumBac...
9,Quantinuum,H2-2SC,False,BackendInfo(name='EmulatorEnabledQuantinuumBac...


In [60]:
for d in QuantinuumBackend.available_devices():

    print(d)

BackendInfo(name='QuantinuumBackend', device_name='H1-1', version='0.59.1', architecture=<tket::FullyConnected, nodes=20>, gate_set={OpType.Measure, OpType.Reset, OpType.PhasedX, OpType.Barrier, OpType.ZZMax, OpType.ZZPhase, OpType.WASM, OpType.SetBits, OpType.CopyBits, OpType.RangePredicate, OpType.ExplicitPredicate, OpType.ExplicitModifier, OpType.MultiBit, OpType.Rz, OpType.TK2, OpType.ClExpr, OpType.RNGSeed, OpType.RNGBound, OpType.RNGIndex, OpType.RNGNum, OpType.JobShotNum}, n_cl_reg=4000, supports_fast_feedforward=True, supports_reset=True, supports_midcircuit_measurement=True, all_node_gate_errors=None, all_edge_gate_errors=None, all_readout_errors=None, averaged_node_gate_errors=None, averaged_edge_gate_errors=None, averaged_readout_errors=None, misc={'wasm': True, 'batching': True, 'supported_languages': ['OPENQASM 2.0', 'QIR 1.0'], 'benchmarks': {'qv': {'date': '2024-04-04', 'value': 1048576.0}}, 'max_classical_register_width': 63, 'syntax_checker': 'H1-1SC', 'n_gate_zones': 

In [62]:
from pytket.extensions.quantinuum import QuantinuumBackend

from pytket import Circuit

for name in ["H2-1SC", "H2-1E", "H2-1", "H2-1LE"]:

    print("\nTrying", name)

    try:

        backend = QuantinuumBackend(name)

        c = Circuit(1, 1)

        c.H(0)

        c.measure_all()

        cc = backend.get_compiled_circuit(c)

        handle = backend.process_circuit(cc, n_shots=1)

        print("submitted:", handle)

    except Exception as e:

        print(type(e), e)


Trying H2-1SC
<class 'pytket.extensions.quantinuum.backends.quantinuum.DeviceNotAvailable'> H2-1SC

Trying H2-1E
<class 'pytket.extensions.quantinuum.backends.quantinuum.DeviceNotAvailable'> H2-1E

Trying H2-1
<class 'NotImplementedError'> Cannot obtain result handles.

Trying H2-1LE
submitted: ('2b29f2ee-6a85-11f1-8c44-beec94cd1c06', 'null', 1, '[["c", 0]]')


In [63]:
for name in ["H2-1SC", "H2-1E", "H2-1"]:

    print("\n====================")

    print("Backend:", name)

    try:

        backend = QuantinuumBackend(name)


        c = Circuit(1, 1)

        c.X(0)

        c.Measure(0, 0)

        cc = backend.get_compiled_circuit(c)

        print("Compiled OK")

        handle = backend.process_circuit(cc, n_shots=1)

        print("Submitted OK:", handle)

    except Exception as e:

        print("FAILED:")

        print(type(e))

        print(e)


Backend: H2-1SC
FAILED:
<class 'pytket.extensions.quantinuum.backends.quantinuum.DeviceNotAvailable'>
H2-1SC

Backend: H2-1E
FAILED:
<class 'pytket.extensions.quantinuum.backends.quantinuum.DeviceNotAvailable'>
H2-1E

Backend: H2-1
Compiled OK
FAILED:
<class 'NotImplementedError'>
Cannot obtain result handles.


In [64]:
qnx.devices.get_all().df()

,backend_name,device_name,nexus_hosted,backend_info
0,Aer,aer_simulator,True,"BackendInfo(name='AerBackend', device_name='ae..."
1,AerState,aer_simulator_statevector,True,"BackendInfo(name='AerStateBackend', device_nam..."
2,AerUnitary,aer_simulator_unitary,True,"BackendInfo(name='AerUnitaryBackend', device_n..."
3,Braket,sv1,True,"BackendInfo(name='BraketBackend', device_name=..."
4,Helios-1E-lite,None,True,"BackendInfo(name='Helios-1E-lite', device_name..."
5,Quantinuum,H2-2E,False,BackendInfo(name='EmulatorEnabledQuantinuumBac...
6,Quantinuum,H2-1E,False,BackendInfo(name='EmulatorEnabledQuantinuumBac...
7,Quantinuum,H2-1SC,False,BackendInfo(name='EmulatorEnabledQuantinuumBac...
8,Quantinuum,H2-1,False,BackendInfo(name='EmulatorEnabledQuantinuumBac...
9,Quantinuum,H2-2SC,False,BackendInfo(name='EmulatorEnabledQuantinuumBac...


In [65]:
config = qnx.QuantinuumConfig(device_name="H2-Emulator")



Backend: H2-1SC
FAILED:
<class 'pytket.extensions.quantinuum.backends.quantinuum.DeviceNotAvailable'>
H2-1SC

Backend: H2-1E
FAILED:
<class 'pytket.extensions.quantinuum.backends.quantinuum.DeviceNotAvailable'>
H2-1E

Backend: H2-1
Compiled OK
FAILED:
<class 'NotImplementedError'>
Cannot obtain result handles.


In [ ]:
project = qnx.projects.get_or_create(name="Nexus-Workflow-Demonstration")
qnx.context.set_active_project(project)

jobname_suffix = datetime.datetime.now().strftime("%Y_%m_%d-%H-%M-%S")

config = qnx.QuantinuumConfig(device_name="H1E")

circuit = Circuit(4)
circuit.H(0)
for i, j in zip(circuit.qubits[:-1], circuit.qubits[1:]):
    circuit.CX(i, j)
circuit.measure_all()

ref = qnx.circuits.upload(circuit=circuit, name=f"GHZ-Circuit-{jobname_suffix}")


ref_compile_job = qnx.start_compile_job(
    programs=[ref],
    backend_config=config,
    optimisation_level=1,
    name=f"compilation-job-{jobname_suffix}",
)


qnx.jobs.wait_for(ref_compile_job)
ref_compiled_circuit = qnx.jobs.results(ref_compile_job)[0].get_output()

ref_execute_job = qnx.start_execute_job(
    programs=[ref_compiled_circuit],
    n_shots=[1],
    backend_config=config,
    name=f"execution-job-{jobname_suffix}",
)

ref_execute_job = qnx.start_execute_job(
    programs=[ref_compiled_circuit],
    n_shots=[1],
    backend_config=config,
    name=f"execution-job-{jobname_suffix}",
)

qnx.jobs.wait_for(ref_execute_job)
ref_result = qnx.jobs.results(ref_execute_job)[0]
backend_result = ref_result.download_result()

UnboundLocalError: No Project set.